In [15]:
# =========================
# TASK 4.6 — STEP 1
# IMPORTS + HELPERS + CORE ASSUMPTIONS
# =========================

import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------
# HELPERS
# -------------------------
def read_table(base_name):
    """
    Reads a file by base name, trying CSV first then XLSX.
    Example:
        read_table("task_4_3_a_market_summary_with_prod")
    """
    csv_path = Path(f"{base_name}.csv")
    xlsx_path = Path(f"{base_name}.xlsx")

    if csv_path.exists():
        return pd.read_csv(csv_path)
    elif xlsx_path.exists():
        return pd.read_excel(xlsx_path)
    else:
        raise FileNotFoundError(f"Could not find {base_name}.csv or {base_name}.xlsx")

def clean_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def safe_dt(s):
    return pd.to_datetime(s, errors="coerce")

# -------------------------
# CURRENCY ASSUMPTIONS
# -------------------------
USD_TO_EUR = 0.92   # keep editable only if your team uses another conversion rate

# -------------------------
# APPENDIX A TRANSPORT COSTS
# -------------------------
US_L_COST_PER_MILE_EUR = 1.75 * USD_TO_EUR   # US Large tractor-trailer
EU_L_COST_PER_KM_EUR = 1.15                  # Europe Large articulated

# Ocean cost assumption from Appendix B
OCEAN_COST_USD_PER_CONTAINER = 2500
OCEAN_COST_EUR_PER_CONTAINER = OCEAN_COST_USD_PER_CONTAINER * USD_TO_EUR

# Savannah DC -> Savannah Port lane assumption
SAVANNAH_DC_TO_PORT_MILES_ONE_WAY = 10

# -------------------------
# APPENDIX B COSTS
# -------------------------
PACKAGING_COST_EUR_PER_UNIT = 15
INVENTORY_HOLDING_RATE = 0.20

DC_STORAGE_COST_PER_PALLET_POSITION = 110
DC_THROUGHPUT_CAPACITY_COST_PER_PEAK_PALLET = 25
DC_INBOUND_HANDLING_COST_PER_PALLET = 4
DC_OUTBOUND_HANDLING_COST_PER_PALLET = 5
DC_PICKING_COST_PER_UNIT = 3

SAVANNAH_FIXED_DC_OVERHEAD_EUR = 900000
EURO_DC_FIXED_OVERHEAD_EUR = 600000

# Fleet fixed annual costs from Appendix B
US_SMALL_FLEET_EUR = 18000
US_MEDIUM_FLEET_EUR = 35000
US_L_TRACTOR_EUR = 95000
US_L_CHASSIS_EUR = 35000

EU_SMALL_FLEET_EUR = 16000
EU_MEDIUM_FLEET_EUR = 32000
EU_L_TRACTOR_EUR = 90000
EU_L_TRAILER_EUR = 30000

# -------------------------
# ROTTERDAM -> EURO DC DISTANCE APPROXIMATION
# -------------------------
# We will derive approximate km later from 4.3(n) drive_hours
ASSUMED_EU_TRUCK_SPEED_KMPH = 80

# -------------------------
# DEFAULT CUSTOMER DELIVERY COSTS
# Appendix B
# -------------------------
METRO_GT_1M_COST = 8
METRO_250K_TO_1M_COST = 10
METRO_LT_250K_COST = 12
NONMETRO_WITHIN_250_COST = 14
NONMETRO_WITHIN_500_COST = 18
NONMETRO_BEYOND_500_COST = 24

# fallback only
DEFAULT_METRO_DELIVERY_COST_EUR_PER_UNIT = 10
DEFAULT_NONMETRO_DELIVERY_COST_EUR_PER_UNIT = 18

# -------------------------
# MODEL CLUSTERS
# Appendix B:
# Cluster 1 => 25% of selling price
# Cluster 2 => 30% of selling price
# -------------------------
MODEL_CLUSTER_MAP = {
    "F10": 1, "F20": 1, "F30": 1, "F50": 1,
    "K10": 1, "K20": 1, "K30": 1, "K50": 1,
    "L20": 1, "L50": 1,
    "S10": 2, "S20": 2, "S30": 2, "S50": 2,
    "W10": 2, "W20": 2, "W30": 2, "W50": 2,
    "X20": 2, "X50": 2
}

# -------------------------
# ACTUAL SELLING PRICES FROM YOUR PRICESHEET.XLSX
# -------------------------
MODEL_PRICE_EUR = {
    "F10": 360,
    "K10": 360,
    "S10": 360,
    "W10": 360,

    "F20": 480,
    "K20": 480,
    "L20": 480,
    "S20": 480,
    "W20": 480,
    "X20": 480,

    "F30": 600,
    "K30": 600,
    "S30": 600,
    "W30": 600,

    "F50": 720,
    "K50": 720,
    "L50": 720,
    "S50": 720,
    "W50": 720,
    "X50": 720
}

model_econ = pd.DataFrame({
    "model": list(MODEL_PRICE_EUR.keys()),
    "selling_price_eur": list(MODEL_PRICE_EUR.values())
})

model_econ["cluster"] = model_econ["model"].map(MODEL_CLUSTER_MAP)

model_econ["production_unit_cost_eur"] = np.where(
    model_econ["cluster"] == 1,
    0.25 * model_econ["selling_price_eur"],
    0.30 * model_econ["selling_price_eur"]
)

model_econ["packaging_unit_cost_eur"] = PACKAGING_COST_EUR_PER_UNIT

print("Step 1 loaded successfully.")
print(model_econ.sort_values("model"))

Step 1 loaded successfully.
   model  selling_price_eur  cluster  production_unit_cost_eur  \
0    F10                360        1                      90.0   
4    F20                480        1                     120.0   
10   F30                600        1                     150.0   
14   F50                720        1                     180.0   
1    K10                360        1                      90.0   
5    K20                480        1                     120.0   
11   K30                600        1                     150.0   
15   K50                720        1                     180.0   
6    L20                480        1                     120.0   
16   L50                720        1                     180.0   
2    S10                360        2                     108.0   
7    S20                480        2                     144.0   
12   S30                600        2                     180.0   
17   S50                720        2            

In [17]:
# =========================
# TASK 4.6 — STEP 2
# READ ALL REQUIRED INPUT FILES
# =========================

# 4.3 core outputs
market = clean_cols(read_table("task_4_3_a_market_summary_with_prod"))
req = clean_cols(read_table("task_4_3_d_replenishment_request_model_level"))
arr = clean_cols(read_table("task_4_3_o_replenishment_arrivals_manifest"))
euro_inv = clean_cols(read_table("task_4_3_p_eurodc_inventory_ledger"))
sav_inv = clean_cols(read_table("task_4_3_p_savannah_inventory_ledger"))
ga_fleet = clean_cols(read_table("task_4_3_r_georgia_peak_active_fleet"))
eu_fleet = clean_cols(read_table("task_4_3_s_europe_peak_active_fleet"))
rotterdam_manifest = clean_cols(read_table("task_4_3_n_rotterdam_to_eurodc_manifest"))

# 4.5 outputs
fleet_45 = clean_cols(read_table("task_4_5_fleet_requirements_by_year"))
eurodc_45 = clean_cols(read_table("task_4_5_eurodc_design_summary"))
sav_45 = clean_cols(read_table("task_4_5_savannah_design_summary"))

# Task 2 outputs
dc_selected = clean_cols(read_table("dc_selected_final (2)"))
dc_open_plan = clean_cols(read_table("dc_open_plan (1)"))
time_matrix = clean_cols(read_table("time_matrix"))
nodes = clean_cols(read_table("nodes (1)"))

# Optional reference sheets you uploaded
price_sheet = clean_cols(read_table("pricesheet"))
dimensions = clean_cols(read_table("dimensions"))

print("All files loaded successfully.\n")

print("market columns:")
print(market.columns.tolist(), "\n")

print("req columns:")
print(req.columns.tolist(), "\n")

print("arr columns:")
print(arr.columns.tolist(), "\n")

print("euro_inv columns:")
print(euro_inv.columns.tolist(), "\n")

print("sav_inv columns:")
print(sav_inv.columns.tolist(), "\n")

print("ga_fleet columns:")
print(ga_fleet.columns.tolist(), "\n")

print("eu_fleet columns:")
print(eu_fleet.columns.tolist(), "\n")

print("rotterdam_manifest columns:")
print(rotterdam_manifest.columns.tolist(), "\n")

print("dc_selected columns:")
print(dc_selected.columns.tolist(), "\n")

print("dc_open_plan columns:")
print(dc_open_plan.columns.tolist(), "\n")

print("time_matrix columns:")
print(time_matrix.columns.tolist(), "\n")

print("nodes columns:")
print(nodes.columns.tolist(), "\n")

print("price_sheet columns:")
print(price_sheet.columns.tolist(), "\n")

print("dimensions columns:")
print(dimensions.columns.tolist(), "\n")

All files loaded successfully.

market columns:
['year', 'market_id', 'market_type', 'country', 'city', 'euro_dc_id', 'model', 'sales_units', 'potential_demand_units', 'lost_sales_units', 'fill_rate', 'Steady_Daily_Production_Rate_99_UnitsPerDay', 'Planned_Daily_Capacity_99_UnitsPerDay', 'Start_Ahead_Days_Previous_Year_99'] 

req columns:
['sim', 'date', 'year', 'euro_dc_id', 'model', 'requested_units', 'Units_Per_Pallet', 'pallets_required'] 

arr columns:
['sim', 'year', 'euro_dc_id', 'euro_dc_city', 'euro_dc_country', 'container_id', 'container_type', 'replenishment_arrival_time', 'replenishment_arrival_date', 'model', 'units_in_container'] 

euro_inv columns:
['sim', 'year', 'date', 'euro_dc_id', 'model', 'inbound_units', 'outbound_units', 'initial_inventory_units', 'begin_inventory_units', 'end_inventory_units'] 

sav_inv columns:
['sim', 'year', 'date', 'model', 'outbound_units', 'inbound_units', 'initial_inventory_units', 'begin_inventory_units', 'end_inventory_units', 'node'] 


In [19]:
# =========================
# TASK 4.6 — STEP 3
# CLEAN AND STANDARDIZE ALL FILES
# =========================

# -------------------------
# MARKET
# -------------------------
market["year"] = safe_num(market["year"]).astype("Int64")
market["market_id"] = market["market_id"].astype(str).str.strip()
market["market_type"] = market["market_type"].astype(str).str.strip().str.lower()
market["country"] = market["country"].astype(str).str.strip()
market["city"] = market["city"].astype(str).str.strip()
market["euro_dc_id"] = market["euro_dc_id"].astype(str).str.strip()
market["model"] = market["model"].astype(str).str.strip()

market["sales_units"] = safe_num(market["sales_units"]).fillna(0)
market["potential_demand_units"] = safe_num(market["potential_demand_units"]).fillna(0)

# Recompute lost sales in units safely
market["lost_sales_units"] = (
    market["potential_demand_units"] - market["sales_units"]
).clip(lower=0)

# -------------------------
# REPLENISHMENT REQUESTS
# -------------------------
req["sim"] = safe_num(req["sim"]).fillna(0).astype(int)
req["year"] = safe_num(req["year"]).astype("Int64")
req["date"] = safe_dt(req["date"])
req["euro_dc_id"] = req["euro_dc_id"].astype(str).str.strip()
req["model"] = req["model"].astype(str).str.strip()

req["requested_units"] = safe_num(req["requested_units"]).fillna(0)
req["Units_Per_Pallet"] = safe_num(req["Units_Per_Pallet"]).replace(0, np.nan)
req["pallets_required"] = safe_num(req["pallets_required"]).fillna(0)

# -------------------------
# REPLENISHMENT ARRIVALS
# -------------------------
arr["sim"] = safe_num(arr["sim"]).fillna(0).astype(int)
arr["year"] = safe_num(arr["year"]).astype("Int64")
arr["euro_dc_id"] = arr["euro_dc_id"].astype(str).str.strip()
arr["euro_dc_city"] = arr["euro_dc_city"].astype(str).str.strip()
arr["euro_dc_country"] = arr["euro_dc_country"].astype(str).str.strip()
arr["container_id"] = arr["container_id"].astype(str).str.strip()
arr["container_type"] = arr["container_type"].astype(str).str.strip()
arr["replenishment_arrival_time"] = safe_dt(arr["replenishment_arrival_time"])
arr["replenishment_arrival_date"] = safe_dt(arr["replenishment_arrival_date"])
arr["model"] = arr["model"].astype(str).str.strip()
arr["units_in_container"] = safe_num(arr["units_in_container"]).fillna(0)

# -------------------------
# EURO DC INVENTORY LEDGER
# -------------------------
euro_inv["sim"] = safe_num(euro_inv["sim"]).fillna(0).astype(int)
euro_inv["year"] = safe_num(euro_inv["year"]).astype("Int64")
euro_inv["date"] = safe_dt(euro_inv["date"])
euro_inv["euro_dc_id"] = euro_inv["euro_dc_id"].astype(str).str.strip()
euro_inv["model"] = euro_inv["model"].astype(str).str.strip()

for c in euro_inv.columns:
    if c not in ["date", "euro_dc_id", "model"]:
        euro_inv[c] = safe_num(euro_inv[c])

euro_end_cols = [c for c in euro_inv.columns if "end_inventory" in c.lower()]
euro_out_cols = [c for c in euro_inv.columns if "outbound" in c.lower()]

if "end_inventory_units" not in euro_inv.columns:
    if len(euro_end_cols) > 0:
        euro_inv["end_inventory_units"] = euro_inv[euro_end_cols[0]].fillna(0)
    else:
        raise ValueError("Could not identify end_inventory_units in euro_inv.")

if "outbound_units" not in euro_inv.columns:
    if len(euro_out_cols) > 0:
        euro_inv["outbound_units"] = euro_inv[euro_out_cols[0]].fillna(0)
    else:
        euro_inv["outbound_units"] = 0

euro_inv["end_inventory_units"] = safe_num(euro_inv["end_inventory_units"]).fillna(0)
euro_inv["outbound_units"] = safe_num(euro_inv["outbound_units"]).fillna(0)

# -------------------------
# SAVANNAH INVENTORY LEDGER
# -------------------------
sav_inv["sim"] = safe_num(sav_inv["sim"]).fillna(0).astype(int)
sav_inv["year"] = safe_num(sav_inv["year"]).astype("Int64")
sav_inv["date"] = safe_dt(sav_inv["date"])
sav_inv["model"] = sav_inv["model"].astype(str).str.strip()
sav_inv["node"] = sav_inv["node"].astype(str).str.strip()

sav_inv["outbound_units"] = safe_num(sav_inv["outbound_units"]).fillna(0)
sav_inv["inbound_units"] = safe_num(sav_inv["inbound_units"]).fillna(0)
sav_inv["initial_inventory_units"] = safe_num(sav_inv["initial_inventory_units"]).fillna(0)
sav_inv["begin_inventory_units"] = safe_num(sav_inv["begin_inventory_units"]).fillna(0)
sav_inv["end_inventory_units"] = safe_num(sav_inv["end_inventory_units"]).fillna(0)

# -------------------------
# GEORGIA FLEET
# -------------------------
ga_fleet["sim"] = safe_num(ga_fleet["sim"]).fillna(0).astype(int)
ga_fleet["year"] = safe_num(ga_fleet["year"]).astype("Int64")
ga_fleet["peak_active_truckers"] = safe_num(ga_fleet["peak_active_truckers"]).fillna(0)
ga_fleet["peak_active_trucks"] = safe_num(ga_fleet["peak_active_trucks"]).fillna(0)
ga_fleet["peak_active_chassis"] = safe_num(ga_fleet["peak_active_chassis"]).fillna(0)

# -------------------------
# EUROPE FLEET
# -------------------------
eu_fleet["sim"] = safe_num(eu_fleet["sim"]).fillna(0).astype(int)
eu_fleet["year"] = safe_num(eu_fleet["year"]).astype("Int64")
eu_fleet["peak_active_truckers"] = safe_num(eu_fleet["peak_active_truckers"]).fillna(0)
eu_fleet["peak_active_trucks"] = safe_num(eu_fleet["peak_active_trucks"]).fillna(0)
eu_fleet["peak_active_chassis"] = safe_num(eu_fleet["peak_active_chassis"]).fillna(0)

# -------------------------
# ROTTERDAM -> EURO DC MANIFEST
# -------------------------
rotterdam_manifest["year"] = safe_num(rotterdam_manifest["year"]).astype("Int64")
rotterdam_manifest["truck_chassis_id"] = rotterdam_manifest["truck_chassis_id"].astype(str).str.strip()
rotterdam_manifest["container_id"] = rotterdam_manifest["container_id"].astype(str).str.strip()
rotterdam_manifest["container_type"] = rotterdam_manifest["container_type"].astype(str).str.strip()
rotterdam_manifest["euro_dc_id"] = rotterdam_manifest["euro_dc_id"].astype(str).str.strip()
rotterdam_manifest["euro_dc_city"] = rotterdam_manifest["euro_dc_city"].astype(str).str.strip()
rotterdam_manifest["euro_dc_country"] = rotterdam_manifest["euro_dc_country"].astype(str).str.strip()

rotterdam_manifest["departure_time_rotterdam_port"] = safe_dt(rotterdam_manifest["departure_time_rotterdam_port"])
rotterdam_manifest["arrival_time_euro_dc"] = safe_dt(rotterdam_manifest["arrival_time_euro_dc"])
rotterdam_manifest["arrival_date_euro_dc"] = safe_dt(rotterdam_manifest["arrival_date_euro_dc"])

rotterdam_manifest["drive_hours"] = safe_num(rotterdam_manifest["drive_hours"]).fillna(0)
rotterdam_manifest["rest_hours"] = safe_num(rotterdam_manifest["rest_hours"]).fillna(0)
rotterdam_manifest["total_transit_hours"] = safe_num(rotterdam_manifest["total_transit_hours"]).fillna(0)
rotterdam_manifest["units_unloaded"] = safe_num(rotterdam_manifest["units_unloaded"]).fillna(0)
rotterdam_manifest["pallets_unloaded"] = safe_num(rotterdam_manifest["pallets_unloaded"]).fillna(0)

# -------------------------
# TASK 4.5 OUTPUTS
# -------------------------
fleet_45["year"] = safe_num(fleet_45["year"]).astype("Int64")

eurodc_45["year"] = safe_num(eurodc_45["year"]).astype("Int64")
eurodc_45["euro_dc_id"] = eurodc_45["euro_dc_id"].astype(str).str.strip()

sav_45["year"] = safe_num(sav_45["year"]).astype("Int64")
if "node" in sav_45.columns:
    sav_45["node"] = sav_45["node"].astype(str).str.strip()

for df in [fleet_45, eurodc_45, sav_45]:
    for c in df.columns:
        if c not in ["year", "euro_dc_id", "node", "main_center_type", "main_processing_activities"]:
            df[c] = safe_num(df[c])

# -------------------------
# TASK 2 OUTPUTS
# -------------------------
for c in dc_selected.columns:
    if c.lower() in ["cand_id", "country", "city"]:
        dc_selected[c] = dc_selected[c].astype(str).str.strip()

for c in dc_open_plan.columns:
    if "cand_id" in c.lower():
        dc_open_plan[c] = dc_open_plan[c].astype(str).str.strip()
    elif "year" in c.lower():
        dc_open_plan[c] = safe_num(dc_open_plan[c])

for c in time_matrix.columns:
    if c.lower() in ["cand_id", "node_id"]:
        time_matrix[c] = time_matrix[c].astype(str).str.strip()
    else:
        time_matrix[c] = safe_num(time_matrix[c])

for c in nodes.columns:
    if c.lower() in ["node_id", "node_type", "country", "cca2", "city"]:
        nodes[c] = nodes[c].astype(str).str.strip()
    else:
        nodes[c] = safe_num(nodes[c])

# -------------------------
# OPTIONAL SHEETS
# -------------------------
for df in [price_sheet, dimensions]:
    df.columns = [str(c).strip() for c in df.columns]

# -------------------------
# QUICK CHECKS
# -------------------------
print("Step 3 cleaning completed.\n")

print("market shape:", market.shape)
print("req shape:", req.shape)
print("arr shape:", arr.shape)
print("euro_inv shape:", euro_inv.shape)
print("sav_inv shape:", sav_inv.shape)
print("ga_fleet shape:", ga_fleet.shape)
print("eu_fleet shape:", eu_fleet.shape)
print("rotterdam_manifest shape:", rotterdam_manifest.shape)
print("dc_selected shape:", dc_selected.shape)
print("dc_open_plan shape:", dc_open_plan.shape)
print("time_matrix shape:", time_matrix.shape)
print("nodes shape:", nodes.shape)

print("\nSample cleaned market rows:")
print(market.head(3))

print("\nSample cleaned req rows:")
print(req.head(3))

print("\nSample cleaned rotterdam_manifest rows:")
print(rotterdam_manifest.head(3))

Step 3 cleaning completed.

market shape: (39320, 14)
req shape: (189920, 8)
arr shape: (198333, 11)
euro_inv shape: (192560, 10)
sav_inv shape: (58440, 10)
ga_fleet shape: (8, 5)
eu_fleet shape: (8, 5)
rotterdam_manifest shape: (57960, 16)
dc_selected shape: (4, 7)
dc_open_plan shape: (4, 8)
time_matrix shape: (30502, 4)
nodes shape: (302, 7)

Sample cleaned market rows:
   year         market_id market_type  country     city     euro_dc_id model  \
0  2027  METRO_BE_antwerp       metro  Belgium  Antwerp  CAND_DE_koeln   F10   
1  2027  METRO_BE_antwerp       metro  Belgium  Antwerp  CAND_DE_koeln   F20   
2  2027  METRO_BE_antwerp       metro  Belgium  Antwerp  CAND_DE_koeln   F30   

   sales_units  potential_demand_units  lost_sales_units  fill_rate  \
0     8.840933                8.840933               0.0        1.0   
1     6.406975                6.406975               0.0        1.0   
2     1.835547                1.835547               0.0        1.0   

   Steady_Daily_Pro

In [20]:
# =========================
# TASK 4.6 — STEP 4
# BUILD MODEL ECONOMICS + ANNUAL COMMERCIAL SUMMARY
# =========================

# -------------------------
# 4A. MODEL ECONOMICS TABLE
# -------------------------
model_econ = pd.DataFrame({
    "model": list(MODEL_PRICE_EUR.keys()),
    "selling_price_eur": list(MODEL_PRICE_EUR.values())
})

model_econ["cluster"] = model_econ["model"].map(MODEL_CLUSTER_MAP)

model_econ["production_unit_cost_eur"] = np.where(
    model_econ["cluster"] == 1,
    0.25 * model_econ["selling_price_eur"],
    0.30 * model_econ["selling_price_eur"]
)

model_econ["packaging_unit_cost_eur"] = PACKAGING_COST_EUR_PER_UNIT

print("Model economics table:")
print(model_econ.sort_values("model"))

# -------------------------
# 4B. ANNUAL COMMERCIAL SUMMARY BY MODEL
# -------------------------
annual_demand_model = (
    market.groupby(["year", "model"], as_index=False)
    .agg(
        expected_demand_units=("potential_demand_units", "sum"),
        realized_sales_units=("sales_units", "sum"),
        lost_sales_units=("lost_sales_units", "sum")
    )
)

annual_demand_model = annual_demand_model.merge(
    model_econ,
    on="model",
    how="left"
)

annual_demand_model["revenue_eur"] = (
    annual_demand_model["realized_sales_units"] *
    annual_demand_model["selling_price_eur"]
)

annual_demand_model["lost_revenue_eur"] = (
    annual_demand_model["lost_sales_units"] *
    annual_demand_model["selling_price_eur"]
)

# -------------------------
# 4C. ANNUAL COMMERCIAL SUMMARY TOTAL
# -------------------------
annual_commercial_summary = (
    annual_demand_model.groupby("year", as_index=False)
    .agg(
        expected_demand_units=("expected_demand_units", "sum"),
        realized_sales_units=("realized_sales_units", "sum"),
        lost_sales_units=("lost_sales_units", "sum"),
        revenue_eur=("revenue_eur", "sum"),
        lost_revenue_eur=("lost_revenue_eur", "sum")
    )
)

annual_commercial_summary["fill_rate"] = np.where(
    annual_commercial_summary["expected_demand_units"] > 0,
    annual_commercial_summary["realized_sales_units"] / annual_commercial_summary["expected_demand_units"],
    np.nan
)

print("\nAnnual commercial summary by model:")
print(annual_demand_model.head())

print("\nAnnual commercial summary total:")
print(annual_commercial_summary)

Model economics table:
   model  selling_price_eur  cluster  production_unit_cost_eur  \
0    F10                360        1                      90.0   
4    F20                480        1                     120.0   
10   F30                600        1                     150.0   
14   F50                720        1                     180.0   
1    K10                360        1                      90.0   
5    K20                480        1                     120.0   
11   K30                600        1                     150.0   
15   K50                720        1                     180.0   
6    L20                480        1                     120.0   
16   L50                720        1                     180.0   
2    S10                360        2                     108.0   
7    S20                480        2                     144.0   
12   S30                600        2                     180.0   
17   S50                720        2                 

In [21]:
# =========================
# TASK 4.6 — STEP 5
# PRODUCTION COST + PACKAGING COST SUMMARY
# =========================

annual_prod_cost_model = annual_demand_model.copy()

# Production cost per Appendix B
annual_prod_cost_model["production_cost_eur"] = (
    annual_prod_cost_model["realized_sales_units"] *
    annual_prod_cost_model["production_unit_cost_eur"]
)

# Packaging cost per Appendix B
annual_prod_cost_model["packaging_cost_eur"] = (
    annual_prod_cost_model["realized_sales_units"] *
    annual_prod_cost_model["packaging_unit_cost_eur"]
)

annual_prod_packaging_summary = (
    annual_prod_cost_model.groupby("year", as_index=False)
    .agg(
        production_cost_eur=("production_cost_eur", "sum"),
        packaging_cost_eur=("packaging_cost_eur", "sum")
    )
)

print("Annual production + packaging by model:")
print(
    annual_prod_cost_model[
        ["year", "model", "realized_sales_units",
         "production_unit_cost_eur", "production_cost_eur",
         "packaging_unit_cost_eur", "packaging_cost_eur"]
    ].head()
)

print("\nAnnual production + packaging summary:")
print(annual_prod_packaging_summary)

Annual production + packaging by model:
   year model  realized_sales_units  production_unit_cost_eur  \
0  2027   F10            929.823655                      90.0   
1  2027   F20            673.837992                     120.0   
2  2027   F30            193.049152                     150.0   
3  2027   F50            439.424867                     180.0   
4  2027   K10            980.549264                      90.0   

   production_cost_eur  packaging_unit_cost_eur  packaging_cost_eur  
0         83684.128924                       15        13947.354821  
1         80860.559046                       15        10107.569881  
2         28957.372808                       15         2895.737281  
3         79096.476020                       15         6591.373002  
4         88249.433739                       15        14708.238957  

Annual production + packaging summary:
   year  production_cost_eur  packaging_cost_eur
0  2027         1.124413e+06        1.286112e+05
1  2028    

In [22]:
# =========================
# TASK 4.6 — STEP 6
# INVENTORY HOLDING COST
# =========================

# 6A. Savannah average inventory by year and model
sav_avg_inv = (
    sav_inv.groupby(["year", "model"], as_index=False)
    .agg(
        avg_inventory_units=("end_inventory_units", "mean")
    )
)

# 6B. Euro DC average inventory by year and model
euro_avg_inv = (
    euro_inv.groupby(["year", "model"], as_index=False)
    .agg(
        avg_inventory_units=("end_inventory_units", "mean")
    )
)

# 6C. Ocean in-transit inventory approximation
# Use annual arrived units as proxy and assume average in-transit inventory = 0.5 * annual arrived units
ocean_inv = (
    arr.groupby(["year", "model"], as_index=False)
    .agg(
        arrived_units=("units_in_container", "sum")
    )
)

ocean_inv["avg_inventory_units"] = 0.5 * ocean_inv["arrived_units"]

# Keep only needed columns
ocean_avg_inv = ocean_inv[["year", "model", "avg_inventory_units"]].copy()

# 6D. Combine Savannah + Euro DC + Ocean inventory positions
all_inv = pd.concat([
    sav_avg_inv.assign(location="Savannah"),
    euro_avg_inv.assign(location="EuroDC"),
    ocean_avg_inv.assign(location="OceanTransit")
], ignore_index=True)

# Merge model economics for unit production value
all_inv = all_inv.merge(
    model_econ[["model", "production_unit_cost_eur"]],
    on="model",
    how="left"
)

# Inventory holding cost = avg inventory units * production value * annual holding rate
all_inv["inventory_holding_cost_eur"] = (
    all_inv["avg_inventory_units"] *
    all_inv["production_unit_cost_eur"] *
    INVENTORY_HOLDING_RATE
)

# 6E. Annual summary
annual_inventory_holding_summary = (
    all_inv.groupby("year", as_index=False)
    .agg(
        inventory_holding_cost_eur=("inventory_holding_cost_eur", "sum")
    )
)

print("Inventory holding cost by location/model:")
print(all_inv.head())

print("\nAnnual inventory holding summary:")
print(annual_inventory_holding_summary)

Inventory holding cost by location/model:
   year model  avg_inventory_units  location  production_unit_cost_eur  \
0  2027   F10          4152.061308  Savannah                      90.0   
1  2027   F20          1768.107292  Savannah                     120.0   
2  2027   F30           988.153756  Savannah                     150.0   
3  2027   F50           317.686818  Savannah                     180.0   
4  2027   K10          2860.826365  Savannah                      90.0   

   inventory_holding_cost_eur  
0                74737.103540  
1                42434.575016  
2                29644.612675  
3                11436.725445  
4                51494.874562  

Annual inventory holding summary:
   year  inventory_holding_cost_eur
0  2027                6.591755e+05
1  2028                1.359634e+06
2  2029                3.090447e+06
3  2030                4.297692e+06
4  2031                5.289197e+06
5  2032                5.737282e+06
6  2033                6.092582e+0

In [23]:
# =========================
# TASK 4.6 — STEP 7
# ROTTERDAM-TO-EURO DC DISTANCE APPROXIMATION + TRANSPORT COST
# =========================

# -------------------------
# 7A. ROTTERDAM -> EURO DC APPROX DISTANCE TABLE
# -------------------------
rotterdam_dc_distance = (
    rotterdam_manifest.groupby(["euro_dc_id", "euro_dc_city", "euro_dc_country"], as_index=False)
    .agg(
        avg_drive_hours=("drive_hours", "mean"),
        avg_total_transit_hours=("total_transit_hours", "mean")
    )
)

rotterdam_dc_distance["approx_distance_km"] = (
    rotterdam_dc_distance["avg_drive_hours"] * ASSUMED_EU_TRUCK_SPEED_KMPH
)

print("Approx Rotterdam -> Euro DC distance table:")
print(rotterdam_dc_distance)

# -------------------------
# 7B. OCEAN TRANSPORT COST
# -------------------------
annual_ocean = (
    arr.groupby("year", as_index=False)
    .agg(
        ocean_containers=("container_id", "nunique")
    )
)

annual_ocean["ocean_transport_cost_eur"] = (
    annual_ocean["ocean_containers"] * OCEAN_COST_EUR_PER_CONTAINER
)

# -------------------------
# 7C. SAVANNAH DC -> SAVANNAH PORT GROUND COST
# -------------------------
# Approximation: each ocean container implies one Savannah-to-port haul
annual_us_ground = annual_ocean.copy()

annual_us_ground["savannah_port_trips"] = annual_us_ground["ocean_containers"]

annual_us_ground["us_ground_transport_cost_eur"] = (
    annual_us_ground["savannah_port_trips"] *
    SAVANNAH_DC_TO_PORT_MILES_ONE_WAY *
    US_L_COST_PER_MILE_EUR
)

# -------------------------
# 7D. ROTTERDAM PORT -> EURO DC GROUND COST
# -------------------------
rotterdam_manifest_cost = rotterdam_manifest.merge(
    rotterdam_dc_distance[["euro_dc_id", "approx_distance_km"]],
    on="euro_dc_id",
    how="left"
)

# One truck-chassis/container move per manifest row
rotterdam_manifest_cost["eu_ground_cost_eur"] = (
    rotterdam_manifest_cost["approx_distance_km"] * EU_L_COST_PER_KM_EUR
)

annual_eu_ground = (
    rotterdam_manifest_cost.groupby("year", as_index=False)
    .agg(
        eu_ground_transport_cost_eur=("eu_ground_cost_eur", "sum")
    )
)

# -------------------------
# 7E. COMBINED ANNUAL TRANSPORT SUMMARY
# -------------------------
annual_transport_summary = annual_ocean[["year", "ocean_transport_cost_eur"]].merge(
    annual_us_ground[["year", "us_ground_transport_cost_eur"]],
    on="year",
    how="outer"
).merge(
    annual_eu_ground,
    on="year",
    how="outer"
).fillna(0)

annual_transport_summary["total_transport_cost_eur"] = (
    annual_transport_summary["ocean_transport_cost_eur"] +
    annual_transport_summary["us_ground_transport_cost_eur"] +
    annual_transport_summary["eu_ground_transport_cost_eur"]
)

print("\nAnnual transport summary:")
print(annual_transport_summary)

Approx Rotterdam -> Euro DC distance table:
       euro_dc_id euro_dc_city euro_dc_country  avg_drive_hours  \
0   CAND_DE_koeln        Koeln         Germany              8.0   
1  CAND_ES_madrid       Madrid           Spain              8.0   
2    CAND_IT_rome         Rome           Italy              8.0   
3    CAND_PL_lodz         Lodz          Poland              8.0   

   avg_total_transit_hours  approx_distance_km  
0                      8.0               640.0  
1                      8.0               640.0  
2                      8.0               640.0  
3                      8.0               640.0  

Annual transport summary:
   year  ocean_transport_cost_eur  us_ground_transport_cost_eur  \
0  2027                 4278000.0                       29946.0   
1  2028                 8753800.0                       61276.6   
2  2029                14209400.0                       99465.8   
3  2030                19584500.0                      137091.5   
4  2031      

In [24]:
# =========================
# TASK 4.6 — STEP 8
# CUSTOMER DELIVERY COST
# =========================

# ---------------------------------------------------
# 8A. BUILD MARKET -> DELIVERY COST LOOKUP
# ---------------------------------------------------
# Logic used:
# 1. If market is metro:
#       assign metro cost by city population tier
# 2. If market is non-metro:
#       assign cost by distance band from assigned Euro DC
#
# IMPORTANT:
# This step assumes your nodes file contains a population-like field
# for metro cities. If not, fallback metro cost = 10 is used.
# ---------------------------------------------------

# Clean task-2 supporting tables
tm = time_matrix.copy()
nd = nodes.copy()
dop = dc_open_plan.copy()

# Standardize possible column names
tm.columns = [str(c).strip() for c in tm.columns]
nd.columns = [str(c).strip() for c in nd.columns]
dop.columns = [str(c).strip() for c in dop.columns]

# Detect dc_open_plan columns
cand_col = [c for c in dop.columns if "cand_id" in c.lower()][0]
open_year_col = [c for c in dop.columns if "open" in c.lower() and "year" in c.lower()][0]

dop = dop[[cand_col, open_year_col]].copy()
dop = dop.rename(columns={cand_col: "cand_id", open_year_col: "first_open_year"})
dop["cand_id"] = dop["cand_id"].astype(str).str.strip()
dop["first_open_year"] = safe_num(dop["first_open_year"]).astype("Int64")

# Detect node id and node type columns
node_id_col = [c for c in nd.columns if c.lower() == "node_id"][0]
node_type_col = [c for c in nd.columns if "node_type" in c.lower()][0]

nd = nd.rename(columns={node_id_col: "node_id", node_type_col: "node_type"})
nd["node_id"] = nd["node_id"].astype(str).str.strip()
nd["node_type"] = nd["node_type"].astype(str).str.strip().str.lower()

# Try to detect city population column in nodes
possible_pop_cols = [c for c in nd.columns if "pop" in c.lower()]
population_col = possible_pop_cols[0] if len(possible_pop_cols) > 0 else None

# Standardize time_matrix columns
tm = tm.rename(columns={
    [c for c in tm.columns if c.lower() == "cand_id"][0]: "cand_id",
    [c for c in tm.columns if c.lower() == "node_id"][0]: "node_id",
    [c for c in tm.columns if "dist" in c.lower()][0]: "dist_km"
})
tm["cand_id"] = tm["cand_id"].astype(str).str.strip()
tm["node_id"] = tm["node_id"].astype(str).str.strip()
tm["dist_km"] = safe_num(tm["dist_km"])

# ---------------------------------------------------
# 8B. MAP EACH MARKET TO ITS ACTIVE CANDIDATE DC BY YEAR
# ---------------------------------------------------
market_delivery = market.copy()

# We already have year, market_id, market_type, euro_dc_id, sales_units
market_delivery["year"] = safe_num(market_delivery["year"]).astype("Int64")
market_delivery["market_id"] = market_delivery["market_id"].astype(str).str.strip()
market_delivery["euro_dc_id"] = market_delivery["euro_dc_id"].astype(str).str.strip()
market_delivery["market_type"] = market_delivery["market_type"].astype(str).str.strip().str.lower()
market_delivery["sales_units"] = safe_num(market_delivery["sales_units"]).fillna(0)

# Assume euro_dc_id aligns with cand_id naming from task 2 / 4
market_delivery["cand_id"] = market_delivery["euro_dc_id"]

# Join opening year
market_delivery = market_delivery.merge(
    dop,
    on="cand_id",
    how="left"
)

# Keep only rows where DC is open by that year, if opening info exists
if market_delivery["first_open_year"].notna().any():
    market_delivery = market_delivery[
        (market_delivery["first_open_year"].isna()) |
        (market_delivery["year"] >= market_delivery["first_open_year"])
    ].copy()

# ---------------------------------------------------
# 8C. ATTACH DISTANCE FROM ACTIVE DC TO MARKET
# ---------------------------------------------------
market_delivery = market_delivery.merge(
    tm[["cand_id", "node_id", "dist_km"]],
    left_on=["cand_id", "market_id"],
    right_on=["cand_id", "node_id"],
    how="left"
)

# Attach node metadata
market_delivery = market_delivery.merge(
    nd,
    left_on="market_id",
    right_on="node_id",
    how="left",
    suffixes=("", "_node")
)

# ---------------------------------------------------
# 8D. ASSIGN DELIVERY COST PER UNIT
# ---------------------------------------------------
def assign_delivery_cost(row):
    market_type = str(row.get("market_type", "")).lower()
    dist_km = row.get("dist_km", np.nan)

    # METRO markets
    if market_type == "metro":
        if population_col is not None and pd.notna(row.get(population_col, np.nan)):
            pop_val = row.get(population_col, np.nan)
            if pop_val > 1_000_000:
                return METRO_GT_1M_COST
            elif pop_val >= 250_000:
                return METRO_250K_TO_1M_COST
            else:
                return METRO_LT_250K_COST
        else:
            # fallback if no population column available
            return DEFAULT_METRO_DELIVERY_COST_EUR_PER_UNIT

    # NON-METRO markets
    if pd.isna(dist_km):
        return DEFAULT_NONMETRO_DELIVERY_COST_EUR_PER_UNIT
    elif dist_km <= 250:
        return NONMETRO_WITHIN_250_COST
    elif dist_km <= 500:
        return NONMETRO_WITHIN_500_COST
    else:
        return NONMETRO_BEYOND_500_COST

market_delivery["delivery_cost_per_unit_eur"] = market_delivery.apply(assign_delivery_cost, axis=1)

market_delivery["customer_delivery_cost_eur"] = (
    market_delivery["sales_units"] * market_delivery["delivery_cost_per_unit_eur"]
)

# ---------------------------------------------------
# 8E. ANNUAL CUSTOMER DELIVERY SUMMARY
# ---------------------------------------------------
annual_customer_delivery_summary = (
    market_delivery.groupby("year", as_index=False)
    .agg(
        customer_delivery_cost_eur=("customer_delivery_cost_eur", "sum")
    )
)

print("Sample market delivery rows:")
show_cols = [
    c for c in [
        "year", "market_id", "market_type", "cand_id",
        "dist_km", "sales_units", "delivery_cost_per_unit_eur",
        "customer_delivery_cost_eur"
    ] if c in market_delivery.columns
]
print(market_delivery[show_cols].head())

print("\nAnnual customer delivery summary:")
print(annual_customer_delivery_summary)

Sample market delivery rows:
   year         market_id market_type        cand_id     dist_km  sales_units  \
0  2027  METRO_BE_antwerp       metro  CAND_DE_koeln  216.873006     8.840933   
1  2027  METRO_BE_antwerp       metro  CAND_DE_koeln  216.873006     6.406975   
2  2027  METRO_BE_antwerp       metro  CAND_DE_koeln  216.873006     1.835547   
3  2027  METRO_BE_antwerp       metro  CAND_DE_koeln  216.873006     4.178132   
4  2027  METRO_BE_antwerp       metro  CAND_DE_koeln  216.873006     9.323241   

   delivery_cost_per_unit_eur  customer_delivery_cost_eur  
0                          10                   88.409328  
1                          10                   64.069745  
2                          10                   18.355465  
3                          10                   41.781318  
4                          10                   93.232412  

Annual customer delivery summary:
   year  customer_delivery_cost_eur
0  2027                1.312122e+05
1  2028          

In [25]:
# =========================
# TASK 4.6 — STEP 9
# DC OPERATING COST
# =========================

# ----------------------------------------
# 9A. BUILD MODEL -> UNITS PER PALLET LOOKUP
# ----------------------------------------
model_pallet = (
    req[["model", "Units_Per_Pallet"]]
    .drop_duplicates()
    .copy()
)

model_pallet["model"] = model_pallet["model"].astype(str).str.strip()
model_pallet["Units_Per_Pallet"] = safe_num(model_pallet["Units_Per_Pallet"]).replace(0, np.nan)

# ----------------------------------------
# 9B. EURO DC ANNUAL INBOUND PALLETS
# from replenishment arrivals manifest
# ----------------------------------------
arr_pallet = arr.merge(
    model_pallet,
    on="model",
    how="left"
)

arr_pallet["annual_inbound_pallets_row"] = np.ceil(
    arr_pallet["units_in_container"] / arr_pallet["Units_Per_Pallet"]
)
arr_pallet["annual_inbound_pallets_row"] = (
    arr_pallet["annual_inbound_pallets_row"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

annual_euro_inbound = (
    arr_pallet.groupby(["year", "euro_dc_id"], as_index=False)
    .agg(
        annual_inbound_pallets=("annual_inbound_pallets_row", "sum")
    )
)

# ----------------------------------------
# 9C. EURO DC ANNUAL OUTBOUND PALLETS + PICKED UNITS
# from replenishment request file
# ----------------------------------------
annual_euro_outbound = (
    req.groupby(["year", "euro_dc_id"], as_index=False)
    .agg(
        annual_outbound_pallets=("pallets_required", "sum"),
        annual_picked_units=("requested_units", "sum")
    )
)

# ----------------------------------------
# 9D. EURO DC OPERATING COST
# ----------------------------------------
eurodc_cost = eurodc_45.merge(
    annual_euro_inbound,
    on=["year", "euro_dc_id"],
    how="left"
).merge(
    annual_euro_outbound,
    on=["year", "euro_dc_id"],
    how="left"
)

for c in ["annual_inbound_pallets", "annual_outbound_pallets", "annual_picked_units"]:
    eurodc_cost[c] = eurodc_cost[c].fillna(0)

eurodc_cost["storage_cost_eur"] = (
    eurodc_cost["required_pallet_positions"] * DC_STORAGE_COST_PER_PALLET_POSITION
)

eurodc_cost["throughput_capacity_cost_eur"] = (
    eurodc_cost["peak_daily_total_pallets"] * DC_THROUGHPUT_CAPACITY_COST_PER_PEAK_PALLET
)

eurodc_cost["inbound_handling_cost_eur"] = (
    eurodc_cost["annual_inbound_pallets"] * DC_INBOUND_HANDLING_COST_PER_PALLET
)

eurodc_cost["outbound_handling_cost_eur"] = (
    eurodc_cost["annual_outbound_pallets"] * DC_OUTBOUND_HANDLING_COST_PER_PALLET
)

eurodc_cost["picking_cost_eur"] = (
    eurodc_cost["annual_picked_units"] * DC_PICKING_COST_PER_UNIT
)

eurodc_cost["fixed_overhead_eur"] = EURO_DC_FIXED_OVERHEAD_EUR

eurodc_cost["dc_operating_cost_eur"] = (
    eurodc_cost["storage_cost_eur"] +
    eurodc_cost["throughput_capacity_cost_eur"] +
    eurodc_cost["inbound_handling_cost_eur"] +
    eurodc_cost["outbound_handling_cost_eur"] +
    eurodc_cost["picking_cost_eur"] +
    eurodc_cost["fixed_overhead_eur"]
)

annual_euro_dc_cost = (
    eurodc_cost.groupby("year", as_index=False)
    .agg(
        euro_dc_operating_cost_eur=("dc_operating_cost_eur", "sum")
    )
)

# ----------------------------------------
# 9E. SAVANNAH ANNUAL INBOUND / OUTBOUND PALLETS + PICKED UNITS
# ----------------------------------------
sav_pallet = sav_inv.merge(
    model_pallet,
    on="model",
    how="left"
)

sav_pallet["annual_inbound_pallets_row"] = np.ceil(
    sav_pallet["inbound_units"] / sav_pallet["Units_Per_Pallet"]
)
sav_pallet["annual_outbound_pallets_row"] = np.ceil(
    sav_pallet["outbound_units"] / sav_pallet["Units_Per_Pallet"]
)

sav_pallet["annual_inbound_pallets_row"] = (
    sav_pallet["annual_inbound_pallets_row"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
sav_pallet["annual_outbound_pallets_row"] = (
    sav_pallet["annual_outbound_pallets_row"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

annual_sav_flow = (
    sav_pallet.groupby("year", as_index=False)
    .agg(
        annual_inbound_pallets=("annual_inbound_pallets_row", "sum"),
        annual_outbound_pallets=("annual_outbound_pallets_row", "sum"),
        annual_picked_units=("outbound_units", "sum")
    )
)

# ----------------------------------------
# 9F. SAVANNAH DC OPERATING COST
# ----------------------------------------
sav_cost = sav_45.merge(
    annual_sav_flow,
    on="year",
    how="left"
)

for c in ["annual_inbound_pallets", "annual_outbound_pallets", "annual_picked_units"]:
    sav_cost[c] = sav_cost[c].fillna(0)

sav_cost["storage_cost_eur"] = (
    sav_cost["required_pallet_positions"] * DC_STORAGE_COST_PER_PALLET_POSITION
)

sav_cost["throughput_capacity_cost_eur"] = (
    sav_cost["peak_daily_total_pallets"] * DC_THROUGHPUT_CAPACITY_COST_PER_PEAK_PALLET
)

sav_cost["inbound_handling_cost_eur"] = (
    sav_cost["annual_inbound_pallets"] * DC_INBOUND_HANDLING_COST_PER_PALLET
)

sav_cost["outbound_handling_cost_eur"] = (
    sav_cost["annual_outbound_pallets"] * DC_OUTBOUND_HANDLING_COST_PER_PALLET
)

sav_cost["picking_cost_eur"] = (
    sav_cost["annual_picked_units"] * DC_PICKING_COST_PER_UNIT
)

sav_cost["fixed_overhead_eur"] = SAVANNAH_FIXED_DC_OVERHEAD_EUR

sav_cost["savannah_dc_operating_cost_eur"] = (
    sav_cost["storage_cost_eur"] +
    sav_cost["throughput_capacity_cost_eur"] +
    sav_cost["inbound_handling_cost_eur"] +
    sav_cost["outbound_handling_cost_eur"] +
    sav_cost["picking_cost_eur"] +
    sav_cost["fixed_overhead_eur"]
)

annual_sav_dc_cost = sav_cost[["year", "savannah_dc_operating_cost_eur"]].copy()

# ----------------------------------------
# 9G. TOTAL DC OPERATING COST
# ----------------------------------------
annual_dc_operating_summary = annual_euro_dc_cost.merge(
    annual_sav_dc_cost,
    on="year",
    how="outer"
).fillna(0)

annual_dc_operating_summary["dc_operating_cost_eur"] = (
    annual_dc_operating_summary["euro_dc_operating_cost_eur"] +
    annual_dc_operating_summary["savannah_dc_operating_cost_eur"]
)

print("Euro DC operating cost detail:")
print(eurodc_cost.head())

print("\nSavannah DC operating cost detail:")
print(sav_cost.head())

print("\nAnnual DC operating summary:")
print(annual_dc_operating_summary)

Euro DC operating cost detail:
   year      euro_dc_id  main_center_type  \
0  2027   CAND_DE_koeln  Regional Euro DC   
1  2028   CAND_DE_koeln  Regional Euro DC   
2  2028    CAND_PL_lodz  Regional Euro DC   
3  2029   CAND_DE_koeln  Regional Euro DC   
4  2029  CAND_ES_madrid  Regional Euro DC   

                          main_processing_activities  peak_inventory_units  \
0  Receiving, putaway, storage, pallet handling, ...            193.905355   
1  Receiving, putaway, storage, pallet handling, ...            354.534164   
2  Receiving, putaway, storage, pallet handling, ...            105.736053   
3  Receiving, putaway, storage, pallet handling, ...            877.630213   
4  Receiving, putaway, storage, pallet handling, ...            291.871658   

   peak_inventory_pallets  peak_daily_inbound_pallets  \
0                    26.0                       129.0   
1                    45.0                       229.0   
2                    14.0                       180.0   
3

In [26]:
# =========================
# TASK 4.6 — STEP 10
# FLEET FIXED COST
# =========================

# ----------------------------------------
# 10A. GEORGIA FLEET FIXED COST
# ----------------------------------------
annual_ga_fleet = (
    ga_fleet.groupby("year", as_index=False)
    .agg(
        georgia_peak_active_truckers=("peak_active_truckers", "max"),
        georgia_peak_active_trucks=("peak_active_trucks", "max"),
        georgia_peak_active_chassis=("peak_active_chassis", "max")
    )
)

annual_ga_fleet["georgia_fleet_fixed_cost_eur"] = (
    annual_ga_fleet["georgia_peak_active_trucks"] * US_L_TRACTOR_EUR +
    annual_ga_fleet["georgia_peak_active_chassis"] * US_L_CHASSIS_EUR
)

# ----------------------------------------
# 10B. EUROPE FLEET FIXED COST
# ----------------------------------------
annual_eu_fleet = (
    eu_fleet.groupby("year", as_index=False)
    .agg(
        europe_peak_active_truckers=("peak_active_truckers", "max"),
        europe_peak_active_trucks=("peak_active_trucks", "max"),
        europe_peak_active_chassis=("peak_active_chassis", "max")
    )
)

annual_eu_fleet["europe_fleet_fixed_cost_eur"] = (
    annual_eu_fleet["europe_peak_active_trucks"] * EU_L_TRACTOR_EUR +
    annual_eu_fleet["europe_peak_active_chassis"] * EU_L_TRAILER_EUR
)

# ----------------------------------------
# 10C. TOTAL FLEET FIXED COST
# ----------------------------------------
annual_fleet_fixed_summary = annual_ga_fleet.merge(
    annual_eu_fleet,
    on="year",
    how="outer"
).fillna(0)

annual_fleet_fixed_summary["fleet_fixed_cost_eur"] = (
    annual_fleet_fixed_summary["georgia_fleet_fixed_cost_eur"] +
    annual_fleet_fixed_summary["europe_fleet_fixed_cost_eur"]
)

print("Annual Georgia fleet summary:")
print(annual_ga_fleet)

print("\nAnnual Europe fleet summary:")
print(annual_eu_fleet)

print("\nAnnual fleet fixed cost summary:")
print(annual_fleet_fixed_summary)

Annual Georgia fleet summary:
   year  georgia_peak_active_truckers  georgia_peak_active_trucks  \
0  2027                            16                          16   
1  2028                            31                          31   
2  2029                            79                          79   
3  2030                           123                         123   
4  2031                           142                         142   
5  2032                           161                         161   
6  2033                           165                         165   
7  2034                           179                         179   

   georgia_peak_active_chassis  georgia_fleet_fixed_cost_eur  
0                           16                       2080000  
1                           31                       4030000  
2                           79                      10270000  
3                          123                      15990000  
4                          142   

In [27]:
# =========================
# TASK 4.6 — STEP 11
# FINAL ANNUAL PROFITABILITY TABLE
# =========================

profitability = (
    annual_commercial_summary
    .merge(annual_prod_packaging_summary, on="year", how="left")
    .merge(annual_inventory_holding_summary, on="year", how="left")
    .merge(annual_transport_summary, on="year", how="left")
    .merge(annual_customer_delivery_summary, on="year", how="left")
    .merge(annual_dc_operating_summary[["year", "dc_operating_cost_eur"]], on="year", how="left")
    .merge(annual_fleet_fixed_summary[["year", "fleet_fixed_cost_eur"]], on="year", how="left")
)

fill_cols = [
    "production_cost_eur",
    "packaging_cost_eur",
    "inventory_holding_cost_eur",
    "ocean_transport_cost_eur",
    "us_ground_transport_cost_eur",
    "eu_ground_transport_cost_eur",
    "total_transport_cost_eur",
    "customer_delivery_cost_eur",
    "dc_operating_cost_eur",
    "fleet_fixed_cost_eur"
]

for c in fill_cols:
    profitability[c] = profitability[c].fillna(0)

profitability["total_cost_eur"] = (
    profitability["production_cost_eur"] +
    profitability["packaging_cost_eur"] +
    profitability["inventory_holding_cost_eur"] +
    profitability["total_transport_cost_eur"] +
    profitability["customer_delivery_cost_eur"] +
    profitability["dc_operating_cost_eur"] +
    profitability["fleet_fixed_cost_eur"]
)

profitability["operating_profit_eur"] = (
    profitability["revenue_eur"] - profitability["total_cost_eur"]
)

profitability["profit_margin"] = np.where(
    profitability["revenue_eur"] > 0,
    profitability["operating_profit_eur"] / profitability["revenue_eur"],
    np.nan
)

profitability["profit_per_realized_unit_eur"] = np.where(
    profitability["realized_sales_units"] > 0,
    profitability["operating_profit_eur"] / profitability["realized_sales_units"],
    np.nan
)

print("Final annual profitability table:")
print(profitability)

Final annual profitability table:
   year  expected_demand_units  realized_sales_units  lost_sales_units  \
0  2027            8709.067559           8574.081677        134.985882   
1  2028           17974.155581          17587.335334        386.820247   
2  2029           48046.666991          46487.178378       1559.488613   
3  2030           75636.271953          73402.961979       2233.309974   
4  2031           91624.754004          88753.306446       2871.447558   
5  2032          100747.669248          97533.407283       3214.261965   
6  2033          108967.531191         105483.122916       3484.408275   
7  2034          117203.574845         113448.404723       3755.170122   

    revenue_eur  lost_revenue_eur  fill_rate  production_cost_eur  \
0  4.149896e+06      6.533381e+04   0.984501         1.124413e+06   
1  8.093164e+06      1.780031e+05   0.978479         2.209040e+06   
2  2.167162e+07      7.270101e+05   0.967542         5.918432e+06   
3  3.429272e+07      1.

In [28]:
# =========================
# TASK 4.6 — STEP 12
# HORIZON SUMMARY
# =========================

horizon_summary = pd.DataFrame([{
    "years_covered": f"{int(profitability['year'].min())}-{int(profitability['year'].max())}",
    "expected_demand_units": profitability["expected_demand_units"].sum(),
    "realized_sales_units": profitability["realized_sales_units"].sum(),
    "lost_sales_units": profitability["lost_sales_units"].sum(),
    "revenue_eur": profitability["revenue_eur"].sum(),
    "lost_revenue_eur": profitability["lost_revenue_eur"].sum(),
    "production_cost_eur": profitability["production_cost_eur"].sum(),
    "packaging_cost_eur": profitability["packaging_cost_eur"].sum(),
    "inventory_holding_cost_eur": profitability["inventory_holding_cost_eur"].sum(),
    "ocean_transport_cost_eur": profitability["ocean_transport_cost_eur"].sum(),
    "us_ground_transport_cost_eur": profitability["us_ground_transport_cost_eur"].sum(),
    "eu_ground_transport_cost_eur": profitability["eu_ground_transport_cost_eur"].sum(),
    "total_transport_cost_eur": profitability["total_transport_cost_eur"].sum(),
    "customer_delivery_cost_eur": profitability["customer_delivery_cost_eur"].sum(),
    "dc_operating_cost_eur": profitability["dc_operating_cost_eur"].sum(),
    "fleet_fixed_cost_eur": profitability["fleet_fixed_cost_eur"].sum(),
    "total_cost_eur": profitability["total_cost_eur"].sum(),
    "operating_profit_eur": profitability["operating_profit_eur"].sum(),
    "profit_margin": (
        profitability["operating_profit_eur"].sum() / profitability["revenue_eur"].sum()
        if profitability["revenue_eur"].sum() > 0 else np.nan
    ),
    "profit_per_realized_unit_eur": (
        profitability["operating_profit_eur"].sum() / profitability["realized_sales_units"].sum()
        if profitability["realized_sales_units"].sum() > 0 else np.nan
    )
}])

print("Horizon summary:")
print(horizon_summary.T)

Horizon summary:
                                             0
years_covered                        2027-2034
expected_demand_units            568909.691371
realized_sales_units             551269.798737
lost_sales_units                  17639.892634
revenue_eur                   258685365.884313
lost_revenue_eur                8277390.209759
production_cost_eur            70146611.643051
packaging_cost_eur               8269046.98105
inventory_holding_cost_eur     33542345.179006
ocean_transport_cost_eur           133308000.0
us_ground_transport_cost_eur          933156.0
eu_ground_transport_cost_eur        42658560.0
total_transport_cost_eur           176899716.0
customer_delivery_cost_eur      8380735.763549
dc_operating_cost_eur           36938643.79242
fleet_fixed_cost_eur                 287540000
total_cost_eur                621717099.359076
operating_profit_eur         -363031733.474763
profit_margin                        -1.403372
profit_per_realized_unit_eur       -658.537

In [29]:
# =========================
# TASK 4.6 — STEP 13
# SAVE OUTPUTS
# =========================

annual_demand_model.to_csv("task_4_6_annual_demand_model.csv", index=False)
annual_commercial_summary.to_csv("task_4_6_annual_commercial_summary.csv", index=False)
annual_prod_packaging_summary.to_csv("task_4_6_annual_prod_packaging_summary.csv", index=False)
annual_inventory_holding_summary.to_csv("task_4_6_annual_inventory_holding_summary.csv", index=False)
rotterdam_dc_distance.to_csv("task_4_6_rotterdam_dc_distance_table.csv", index=False)
annual_transport_summary.to_csv("task_4_6_annual_transport_summary.csv", index=False)
annual_customer_delivery_summary.to_csv("task_4_6_annual_customer_delivery_summary.csv", index=False)
annual_dc_operating_summary.to_csv("task_4_6_annual_dc_operating_summary.csv", index=False)
annual_fleet_fixed_summary.to_csv("task_4_6_annual_fleet_fixed_summary.csv", index=False)
profitability.to_csv("task_4_6_profitability_summary.csv", index=False)
horizon_summary.to_csv("task_4_6_horizon_summary.csv", index=False)

print("Saved successfully:")
print("1. task_4_6_annual_demand_model.csv")
print("2. task_4_6_annual_commercial_summary.csv")
print("3. task_4_6_annual_prod_packaging_summary.csv")
print("4. task_4_6_annual_inventory_holding_summary.csv")
print("5. task_4_6_rotterdam_dc_distance_table.csv")
print("6. task_4_6_annual_transport_summary.csv")
print("7. task_4_6_annual_customer_delivery_summary.csv")
print("8. task_4_6_annual_dc_operating_summary.csv")
print("9. task_4_6_annual_fleet_fixed_summary.csv")
print("10. task_4_6_profitability_summary.csv")
print("11. task_4_6_horizon_summary.csv")

Saved successfully:
1. task_4_6_annual_demand_model.csv
2. task_4_6_annual_commercial_summary.csv
3. task_4_6_annual_prod_packaging_summary.csv
4. task_4_6_annual_inventory_holding_summary.csv
5. task_4_6_rotterdam_dc_distance_table.csv
6. task_4_6_annual_transport_summary.csv
7. task_4_6_annual_customer_delivery_summary.csv
8. task_4_6_annual_dc_operating_summary.csv
9. task_4_6_annual_fleet_fixed_summary.csv
10. task_4_6_profitability_summary.csv
11. task_4_6_horizon_summary.csv


In [32]:
# =========================
# TASK 4.7 — STEP 1
# FUEL + CO2 ASSUMPTIONS FROM APPENDIX A
# =========================

# US Large Tractor-Trailer midpoint assumptions
US_L_MPG = 6.5
US_L_CO2_G_PER_MILE = 1650

# Europe Large Articulated midpoint assumptions
EU_L_FUEL_L_PER_100KM = 30.5
EU_L_CO2_G_PER_KM = 875

# Conversion
GALLON_TO_LITER = 3.78541

print("4.7 Step 1 loaded successfully.")
print("US_L_MPG =", US_L_MPG)
print("US_L_CO2_G_PER_MILE =", US_L_CO2_G_PER_MILE)
print("EU_L_FUEL_L_PER_100KM =", EU_L_FUEL_L_PER_100KM)
print("EU_L_CO2_G_PER_KM =", EU_L_CO2_G_PER_KM)

4.7 Step 1 loaded successfully.
US_L_MPG = 6.5
US_L_CO2_G_PER_MILE = 1650
EU_L_FUEL_L_PER_100KM = 30.5
EU_L_CO2_G_PER_KM = 875


In [33]:
# =========================
# TASK 4.7 — STEP 2
# ANNUAL US TRUCKING ACTIVITY
# =========================

# Use same logic as 4.6:
# one Savannah DC -> Savannah Port truck trip per ocean container

annual_us_activity = annual_ocean.copy()

annual_us_activity["us_trips"] = annual_us_activity["ocean_containers"]

annual_us_activity["us_total_miles"] = (
    annual_us_activity["us_trips"] * SAVANNAH_DC_TO_PORT_MILES_ONE_WAY
)

print("Annual US trucking activity:")
print(annual_us_activity)

Annual US trucking activity:
   year  ocean_containers  ocean_transport_cost_eur  us_trips  us_total_miles
0  2027              1860                 4278000.0      1860           18600
1  2028              3806                 8753800.0      3806           38060
2  2029              6178                14209400.0      6178           61780
3  2030              8515                19584500.0      8515           85150
4  2031              8773                20177900.0      8773           87730
5  2032              9561                21990300.0      9561           95610
6  2033              9388                21592400.0      9388           93880
7  2034              9879                22721700.0      9879           98790
